In [1]:
import cv2 
import numpy
import os
import datetime
import face_recognition as fr
import csv

c:\Users\tnsun\AppData\Local\Programs\Python\Python312\Lib\site-packages\face_recognition_models\__init__.py:7: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import resource_filename


In [2]:
path = "images"

In [3]:
os.listdir(path)

['ABHINAV.jpeg', 'agnel.jpeg', 'jeswin.JPG']

In [4]:
mylist = os.listdir(path)

In [5]:
imgs = []
classnames = []

In [6]:
for i in mylist:
    imgpath = os.path.join(path,i)
    current_img = cv2.imread(imgpath)
    imgs.append(current_img)
    classnames.append(i.split('.')[0])
print(classnames)

['ABHINAV', 'agnel', 'jeswin']


In [7]:
def face_encodings(images):
    encodeList = []
    for img in imgs:
        img = cv2.cvtColor(img,cv2.COLOR_BGR2RGB)
        face_loc = fr.face_locations(img)
        # print(face_loc)
        face_enc = fr.face_encodings(img,face_loc)[0]      # we use[0] because we only need the 0th index of the array
        encodeList.append(face_enc)
    return encodeList

In [8]:
encodeList_known_face = face_encodings(imgs)
print(encodeList_known_face)

[array([-0.08995716,  0.12662993,  0.05480674, -0.02835163,  0.01845722,
       -0.04331573, -0.12838706, -0.04885621,  0.14886704, -0.07304891,
        0.17240737,  0.06176393, -0.22306716, -0.15903074,  0.01864525,
        0.09041886, -0.10224761, -0.17569928, -0.05649057, -0.09381309,
       -0.02332733,  0.08103871,  0.01742622,  0.05442105, -0.14779292,
       -0.38607839, -0.0811013 , -0.11486761, -0.02433123, -0.08568132,
       -0.07949126,  0.03149139, -0.20080143, -0.05230197,  0.03091568,
        0.07591988, -0.01659239,  0.01354369,  0.13748974,  0.09795908,
       -0.12157489, -0.03448295,  0.01824534,  0.34129497,  0.19300158,
        0.05562758,  0.04717133, -0.01712045,  0.0946656 , -0.18946023,
        0.09418909,  0.1548195 ,  0.11485865,  0.0390223 ,  0.11269001,
       -0.12964992, -0.04751392,  0.14829396, -0.18881629,  0.09921115,
       -0.0189703 , -0.0818862 , -0.08286574, -0.04585902,  0.27444518,
        0.14512101, -0.15386216, -0.1052676 ,  0.13820854, -0.1

In [9]:
name1 = []
video = cv2.VideoCapture(0)
while True:
    success,frame = video.read()
    frame1 = cv2.cvtColor(frame,cv2.COLOR_BGR2RGB)
    face_loc = fr.face_locations(frame1)
    # print(face_loc)
    face_encr = fr.face_encodings(frame1,face_loc)              # we are not using [0] because we need to identify more than one celebraty in a single image
    
    for encoder ,face_location in zip(face_encr,face_loc):
        match = fr.compare_faces(encodeList_known_face,encoder)
        # print(match)
        # face distance
        distance = fr.face_distance(encodeList_known_face,encoder)
        # print(distance)
        index = numpy.argmin(distance)
        # print(index)
        if match[index]:
            name = classnames[index]
            # print(name)
        else:
            name = "Unknown person"
            print(name)
        y1,x2,y2,x1 = face_location
        cv2.rectangle(frame,(x1,y1),(x2,y2),(0,0,255),2)
        
        if name not in name1:
            name1.append(name)
            name1.append('attendance.csv')
            
        else:
            pass


    cv2.imshow("face",frame)
    if cv2.waitKey(1) & 0XFF == ord('q'):
        break
# print(name1,datetime.datetime.now())
# name1.append('attendance.csv')
video.release()
cv2.destroyAllWindows()

In [10]:
# print(name1,datetime.datetime.now())


In [ ]:
vdo = cv2.VideoCapture(0)

while True :

    succ, img = vdo.read()
    if not succ:
        break
    img1 = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

    face_in_frame = fr.face_locations(img1)
    face_encoder = fr.face_encodings(img1, face_in_frame)

    for encoder, faceloc in zip(face_encoder, face_in_frame) :

        match = fr.compare_faces(imgs, encoder)

        distance = fr.face_distance(imgs, encoder)
        index = numpy.argmin(distance)

        if match[index]:
            name = classnames[index]
            
            # Check if already marked for today
            today = datetime.date.today().isoformat()
            csv_file = 'attendance.csv'
            already_present = False
            try:
                with open(csv_file, 'r') as f:
                    reader = csv.reader(f)
                    for row in reader:
                        if len(row) >= 2 and row[0] == name and row[1] == today:
                            already_present = True
                            break
            except FileNotFoundError:
                pass  # File doesn't exist, so not present
            
            if not already_present:
                with open(csv_file, 'a', newline='') as f:
                    writer = csv.writer(f)
                    now = datetime.datetime.now()
                    writer.writerow([name, today, now.strftime('%H:%M:%S')])
                print(f"Attendance marked for {name}")
            else:
                print(f"{name} already marked today")
        else :
            name = 'Unknown'
            
        # Draw rectangle and name
        y1, x2, y2, x1 = faceloc
        cv2.rectangle(img, (x1, y1), (x2, y2), (0, 255, 0), 2)
        cv2.putText(img, name, (x1, y1 - 10), cv2.FONT_HERSHEY_SIMPLEX, 0.9, (0, 255, 0), 2)

    cv2.imshow('Attendance', img)
    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

vdo.release()
cv2.destroyAllWindows()

ValueError: setting an array element with a sequence. The requested array has an inhomogeneous shape after 1 dimensions. The detected shape was (3,) + inhomogeneous part.

: 